In [1]:
import getml
import mlflow
import getml_mlflow

In [2]:
mlflow.set_tracking_uri("http://localhost:5000")
getml_mlflow.autolog()

In [3]:
getml.set_project("interstate94")

Output()

Connected to project 'interstate94'.

In [4]:
traffic = getml.datasets.load_interstate94(roles=False, units=False)

In [5]:
traffic.set_role("ds", getml.data.roles.time_stamp)
traffic.set_role("holiday", getml.data.roles.categorical)
traffic.set_role("traffic_volume", getml.data.roles.target)

In [6]:
split = getml.data.split.time(traffic, "ds", test=getml.data.time.datetime(2018, 3, 15))

In [7]:
time_series = getml.data.TimeSeries(
    population=traffic,
    split=split,
    time_stamps="ds",
    horizon=getml.data.time.hours(1),
    memory=getml.data.time.days(7),
    lagged_targets=True,
)

pipe = getml.pipeline.Pipeline(
    tags=["memory: 7d", "horizon: 1h", "fast_prop"],
    data_model=time_series.data_model,
    preprocessors=[getml.preprocessors.Seasonal()],
    feature_learners=[
        getml.feature_learning.FastProp(
            loss_function=getml.feature_learning.loss_functions.SquareLoss,
            num_threads=1,
            num_features=20,
        )
    ],
    predictors=[getml.predictors.XGBoostRegressor()],
)
pipe

Pipeline(data_model='population',
         feature_learners=['FastProp'],
         feature_selectors=[],
         include_categorical=False,
         loss_function='SquareLoss',
         peripheral=['traffic'],
         predictors=['XGBoostRegressor'],
         preprocessors=['Seasonal'],
         share_selected_features=0.5,
         tags=['memory: 7d', 'horizon: 1h', 'fast_prop'])

In [8]:
fit1 = pipe.fit(time_series.train)
print(fit1.id, pipe.id)

2025-01-29 18:42:52,609 WARNING getML: Engine metrics are available in the Enterprise edition. Visit https://getml.com/latest/enterprise/ for more information
2025-01-29 18:42:52,613 WARNING getML: Engine metrics are available in the Enterprise edition. Visit https://getml.com/latest/enterprise/ for more information


Checking data model...

Output()

OK.

Output()

Trained pipeline.

2025/01/29 18:43:01 INFO mlflow.tracking._tracking_service.client: 🏃 View run fit at: http://localhost:5000/#/experiments/844494434965818253/runs/8f52d80c2aa2431cafa61cd94f02c201.
2025/01/29 18:43:01 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://localhost:5000/#/experiments/844494434965818253.
2025/01/29 18:43:01 INFO mlflow.tracking._tracking_service.client: 🏃 View run Pipeline-s8SRpu at: http://localhost:5000/#/experiments/844494434965818253/runs/4a2ee58d422347a1a815da5f4c600df6.
2025/01/29 18:43:01 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://localhost:5000/#/experiments/844494434965818253.


Time taken: 0:00:08.393809.

s8SRpu s8SRpu


In [9]:
fit2 = pipe.fit(time_series.train)
print(fit2.id, fit1.id, pipe.id)

2025-01-29 18:43:01,508 WARNING getML: Engine metrics are available in the Enterprise edition. Visit https://getml.com/latest/enterprise/ for more information
2025-01-29 18:43:01,512 WARNING getML: Engine metrics are available in the Enterprise edition. Visit https://getml.com/latest/enterprise/ for more information


Checking data model...

Output()

OK.

Output()

Trained pipeline.

2025/01/29 18:43:02 INFO mlflow.tracking._tracking_service.client: 🏃 View run fit at: http://localhost:5000/#/experiments/844494434965818253/runs/56e8befab9bd437ea2c7709ff54224da.
2025/01/29 18:43:02 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://localhost:5000/#/experiments/844494434965818253.
2025/01/29 18:43:02 INFO mlflow.tracking._tracking_service.client: 🏃 View run Pipeline-EwBj43 at: http://localhost:5000/#/experiments/844494434965818253/runs/d4cd3f8986aa4b21ac8e66a33d89c038.
2025/01/29 18:43:02 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://localhost:5000/#/experiments/844494434965818253.


Time taken: 0:00:00.267125.

EwBj43 EwBj43 EwBj43


In [10]:
pipe.score(time_series.test)

Output()

2025/01/29 18:43:02 INFO mlflow.tracking._tracking_service.client: 🏃 View run score at: http://localhost:5000/#/experiments/844494434965818253/runs/7316b0bda9d14c2b9715e3ec190e160a.
2025/01/29 18:43:02 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://localhost:5000/#/experiments/844494434965818253.


,date time,set used,target,mae,rmse,rsquared
0,2025-01-29 18:43:02,train,traffic_volume,200.4302,299.2045,0.9768
1,2025-01-29 18:43:02,test,traffic_volume,179.9515,269.631,0.9816


In [11]:
pipe.predict(population_table=traffic, peripheral_tables=[traffic])

Output()

2025/01/29 18:43:02 INFO mlflow.tracking._tracking_service.client: 🏃 View run predict at: http://localhost:5000/#/experiments/844494434965818253/runs/6ce4fe7073684ac19c028351d71b932b.
2025/01/29 18:43:02 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://localhost:5000/#/experiments/844494434965818253.


array([[ 592.84667969],
       [1517.38391113],
       [1517.38391113],
       ...,
       [2516.60766602],
       [1748.44812012],
       [1240.2791748 ]])

In [12]:
pipe.transform(population_table=traffic, peripheral_tables=[traffic])

Output()

2025/01/29 18:43:03 INFO mlflow.tracking._tracking_service.client: 🏃 View run transform at: http://localhost:5000/#/experiments/844494434965818253/runs/521b73d784104c97aa7fad9120c125ca.
2025/01/29 18:43:03 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://localhost:5000/#/experiments/844494434965818253.


array([[     0.        ,      0.        ,      0.        , ...,
             0.        ,      0.        ,      0.        ],
       [  1513.        ,   1513.        ,      0.        , ...,
          3600.        ,      0.        ,      0.        ],
       [  1513.        ,   1550.        ,      0.        , ...,
          5400.        ,   1800.        ,   1800.        ],
       ...,
       [  2346.        ,   2781.        ,  18844.        , ...,
        109800.        , 186850.63553545, 106200.        ],
       [  1635.        ,   2159.        ,  13894.        , ...,
         88200.        , 156751.65070901,  84600.        ],
       [   934.        ,   1450.        ,  12571.        , ...,
         66600.        , 114631.40930827,  63000.        ]])

In [13]:
pipe.id

'EwBj43'

In [14]:
# mlflow.data.dataset_source_registry.resolve_dataset_source("traffic.train.parquet")

In [15]:
ds = mlflow.data.dataset_source_registry.resolve_dataset_source("traffic.train.parquet")
print(ds)
print(ds.to_dict())
print(mlflow.get_artifact_uri("traffic.train.parquet"))

{'uri': 'traffic.train.parquet'}
mlflow-artifacts:/0/b2a35eabd17c441888cf21f5be5cba09/artifacts/traffic.train.parquet


/home/manuel/Projects/github/getml-mlflow/.venv/lib/python3.11/site-packages/mlflow/data/dataset_source_registry.py:149: UserWarning: Failed to determine whether UCVolumeDatasetSource can resolve source information for 'traffic.train.parquet'. Exception: 
  return _dataset_source_registry.resolve(
/home/manuel/Projects/github/getml-mlflow/.venv/lib/python3.11/site-packages/mlflow/data/dataset_source_registry.py:149: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(


In [16]:
# run = mlflow.last_active_run()
# run.info.run_name = "ÄÄÄ"

In [17]:
pipe._mlflow_run_info

<RunInfo: artifact_uri='mlflow-artifacts:/844494434965818253/d4cd3f8986aa4b21ac8e66a33d89c038/artifacts', end_time=None, experiment_id='844494434965818253', lifecycle_stage='active', run_id='d4cd3f8986aa4b21ac8e66a33d89c038', run_name='Pipeline-s8SRpu', run_uuid='d4cd3f8986aa4b21ac8e66a33d89c038', start_time=1738172581392, status='RUNNING', user_id='unknown'>